# Stage 3: Baseline Recommendation Systems (Beginner's Guide)
### E-Commerce Intelligence & Decision Support Platform

---

### Why do we need "Baselines" first?
Before training fancy Machine Learning or Deep Learning models, we **must establish baseline benchmarks**:
1. **Level 0 – Random Baseline:** What if we simply guess 5 random products?
2. **Level 1 – Popularity Baseline:** What if we recommend the same 5 bestsellers to everyone?

If our ML model cannot beat Level 1 Popularity, it is useless in practice!

---

### What is Hit-Rate@5?
Imagine we recommend **5 products** to a customer.
We check what that customer **actually bought** in the next 6 months (June–December 2011).
- **HIT:** The customer bought **at least 1** of our 5 recommended products.
- **MISS:** The customer bought none of the 5.

$$\text{Hit-Rate@5} = \frac{\text{Number of Customers with } \ge 1 \text{ Hit}}{\text{Total Customers Tested}}$$


### Step 1: Load Historical & Evaluation Transactions

In [1]:
import pandas as pd
import numpy as np
import os

hist_path = "data/processed/historical_transactions.csv"
if not os.path.exists(hist_path):
    hist_path = "../data/processed/historical_transactions.csv"

eval_path = "data/processed/evaluation_transactions.csv"
if not os.path.exists(eval_path):
    eval_path = "../data/processed/evaluation_transactions.csv"

historical = pd.read_csv(hist_path, dtype={"InvoiceNo": str, "StockCode": str})
evaluation = pd.read_csv(eval_path, dtype={"InvoiceNo": str, "StockCode": str})

print(f"Historical transactions : {len(historical):,} rows")
print(f"Evaluation transactions : {len(evaluation):,} rows")


Historical transactions : 530,472 rows
Evaluation transactions : 248,953 rows


### Step 2: Identify Eligible Customers & Ground Truth
- **Eligible Customers:** Customers who bought in the past (historical) AND came back to shop in the evaluation period.
- **Ground Truth:** The exact set of products each customer actually bought in June–Dec 2011.


In [2]:
hist_customers = set(historical["CustomerID"].unique())
eval_customers = set(evaluation["CustomerID"].unique())

# Customers present in BOTH periods
eligible_customers = sorted(hist_customers & eval_customers)
print(f"Total Eligible Customers for evaluation: {len(eligible_customers):,}")

# Build Ground Truth: {CustomerID: set of StockCodes actually purchased}
eval_eligible = evaluation[evaluation["CustomerID"].isin(eligible_customers)]
ground_truth = eval_eligible.groupby("CustomerID")["StockCode"].apply(set).to_dict()

# Example: inspect what Customer 12347 actually bought in evaluation
sample_cust = eligible_customers[1]
print(f"Example: Customer {sample_cust} bought {len(ground_truth[sample_cust])} distinct products in evaluation.")


Total Eligible Customers for evaluation: 2,602


Example: Customer 12348 bought 3 distinct products in evaluation.


### Step 3: Level 0 – Random Baseline
Let's select the **Top 500 eligible products** as our catalog pool.
For each customer, we randomly recommend 5 products from this pool.


In [3]:
# Top 500 products by historical order count
top_500_pool = (
    historical.groupby("StockCode")["InvoiceNo"]
    .nunique()
    .sort_values(ascending=False)
    .head(500)
    .index.tolist()
)

# Set random seed so results are 100% reproducible
rng = np.random.RandomState(42)

# Generate random 5 recommendations for each customer
random_hits = 0
for cust_id in eligible_customers:
    random_recs = rng.choice(top_500_pool, size=5, replace=False)
    actual_purchases = ground_truth[cust_id]
    
    # Check if any recommendation is in actual purchases
    if set(random_recs) & actual_purchases:
        random_hits += 1

random_hit_rate = (random_hits / len(eligible_customers)) * 100
print(f"Level 0 (Random Baseline) Hit-Rate@5: {random_hit_rate:.2f}% ({random_hits}/{len(eligible_customers)} customers)")


Level 0 (Random Baseline) Hit-Rate@5: 18.10% (471/2602 customers)


### Step 4: Level 1 – Popularity Baseline
Instead of guessing randomly, what if we find the **Top 5 Bestsellers** from historical data and recommend those same 5 products to **every single customer**?


In [4]:
# Find top 5 products by historical order count
top_5_popular = (
    historical.groupby("StockCode")["InvoiceNo"]
    .nunique()
    .sort_values(ascending=False)
    .head(5)
)

print("Top 5 Bestsellers in Historical Data:")
for rank, (code, count) in enumerate(top_5_popular.items(), 1):
    desc = historical[historical["StockCode"] == code]["Description"].mode().iloc[0]
    print(f"  #{rank}: StockCode {code} ({count:,} orders) - {desc}")

top_5_items = set(top_5_popular.index)

# Evaluate Hit-Rate@5 for Popularity
pop_hits = 0
for cust_id in eligible_customers:
    actual_purchases = ground_truth[cust_id]
    if top_5_items & actual_purchases:
        pop_hits += 1

pop_hit_rate = (pop_hits / len(eligible_customers)) * 100
print()
print(f"Level 1 (Popularity Baseline) Hit-Rate@5: {pop_hit_rate:.2f}% ({pop_hits}/{len(eligible_customers)} customers)")


Top 5 Bestsellers in Historical Data:
  #1: StockCode 85123A (3,916 orders) - WHITE HANGING HEART T-LIGHT HOLDER
  #2: StockCode 22423 (2,464 orders) - REGENCY CAKESTAND 3 TIER
  #3: StockCode 85099B (2,252 orders) - JUMBO BAG RED RETROSPOT


  #4: StockCode 21212 (1,978 orders) - PACK OF 72 RETRO SPOT CAKE CASES
  #5: StockCode 84879 (1,855 orders) - ASSORTED COLOUR BIRD ORNAMENT

Level 1 (Popularity Baseline) Hit-Rate@5: 49.92% (1299/2602 customers)


### Step 5: Summary & Key Takeaways
Let's compare the two baselines side-by-side:


In [5]:
comparison = pd.DataFrame([
    {"Baseline": "Level 0 - Random", "Hit-Rate@5": f"{random_hit_rate:.2f}%", "Customers with Hits": random_hits, "Total Customers": len(eligible_customers)},
    {"Baseline": "Level 1 - Popularity", "Hit-Rate@5": f"{pop_hit_rate:.2f}%", "Customers with Hits": pop_hits, "Total Customers": len(eligible_customers)}
])
print(comparison.to_string(index=False))

improvement = ((pop_hit_rate - random_hit_rate) / random_hit_rate) * 100
print()
print(f"Popularity beats Random by +{improvement:.1f}%!")
print("Our goal in Stage 4: Build a Machine Learning model that beats the 49.92% Popularity baseline!")


            Baseline Hit-Rate@5  Customers with Hits  Total Customers
    Level 0 - Random     18.10%                  471             2602
Level 1 - Popularity     49.92%                 1299             2602

Popularity beats Random by +175.8%!
Our goal in Stage 4: Build a Machine Learning model that beats the 49.92% Popularity baseline!
